<a href="https://colab.research.google.com/github/GuilhermeOrnellas/exerc-cio-Pandas./blob/main/Desenvolvendo_um_Sistema_de_RH_com_SQLAlchemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nivel 1

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///sistema_rh.db')

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    """))

novo_funcionario = {"nome": "Carlos Silva", "cargo": "Desenvolvedor Júnior", "salario": 4000.0}

with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
        novo_funcionario
    )

with engine.connect() as conn:
    df_funcionarios = pd.read_sql_query(text("SELECT * FROM funcionarios"), conn)

Nivel 2

In [2]:
import pandas as pd
from sqlalchemy import create_engine, Table, MetaData, Column, Integer, String, Float, insert, update, select, func

engine = create_engine('sqlite:///sistema_rh.db')
metadata = MetaData()

projetos = Table(
    'projetos',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome', String, nullable=False),
    Column('orcamento', Float, nullable=False)
)
metadata.create_all(engine)

funcionarios = Table('funcionarios', metadata, autoload_with=engine)

novos_projetos = [
    {"nome": "Redesenho do App", "orcamento": 50000.0},
    {"nome": "Migração para Nuvem", "orcamento": 120000.0},
    {"nome": "Automação de Testes", "orcamento": 30000.0}
]

with engine.begin() as conn:
    conn.execute(insert(projetos), novos_projetos)

with engine.begin() as conn:
    stmt_update = (
        update(funcionarios)
        .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios.c.salario * 1.10)
    )
    conn.execute(stmt_update)

with engine.connect() as conn:
    stmt_relatorio = (
        select(
            funcionarios.c.cargo,
            func.avg(funcionarios.c.salario).label('media_salarial')
        )
        .group_by(funcionarios.c.cargo)
    )
    df_relatorio = pd.read_sql_query(stmt_relatorio, conn)

Nível 3

In [3]:
from sqlalchemy import create_engine, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker

engine = create_engine('sqlite:///sistema_rh.db')

class Base(DeclarativeBase):
    pass

class Departamento(Base):
    __tablename__ = 'departamentos'

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(nullable=False)

    funcionarios: Mapped[list["FuncionarioORM"]] = relationship(back_populates="departamento")

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(nullable=False)
    cargo: Mapped[str] = mapped_column(nullable=False)
    salario: Mapped[float] = mapped_column(nullable=False)
    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

    departamento: Mapped["Departamento"] = relationship(back_populates="funcionarios")

Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
sessao = Session()

depto_ti = Departamento(nome="TI")
dev1 = FuncionarioORM(nome="Ana Souza", cargo="Desenvolvedora Senior", salario=9000.0, departamento=depto_ti)
dev2 = FuncionarioORM(nome="Bruno Lima", cargo="Pleno Backend", salario=6500.0, departamento=depto_ti)

sessao.add(depto_ti)
sessao.commit()

stmt = (
    select(FuncionarioORM)
    .join(FuncionarioORM.departamento)
    .where(Departamento.nome == "TI")
)

funcionarios_ti = sessao.execute(stmt).scalars().all()

sessao.close()